# Colab Drive I/O diagnostic — SynthHuman

Notebook ini menguji apakah Colab masih bisa membaca file yang sudah ada di Google Drive. Ia hanya membuka path sampel yang ditentukan, tidak memindai seluruh folder, tidak memindahkan file, dan tidak mengunduh dataset.

Jalankan cell berurutan. Jika berhasil, kirim output cell 2–4 untuk menentukan apakah pilot training dari Drive layak dicoba.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import time
import os

SYNTHHUMAN_ROOT = Path('/content/drive/MyDrive/AppleCapstoneRelighting/dataset/sources/SynthHuman')
SAMPLE_ID = '0048517'  # Sampel yang sebelumnya terlihat ada di folder Drive
print('Folder dapat diakses:', SYNTHHUMAN_ROOT.is_dir())
print('Path sampel:', SYNTHHUMAN_ROOT / f'rgb_{SAMPLE_ID}.png')

In [ ]:
# Periksa metadata hanya untuk empat path sampel yang diketahui.
# Jangan gunakan glob(), rglob(), os.listdir(), atau iterdir() pada folder dataset ini.
modalities = {
    'rgb': f'rgb_{SAMPLE_ID}.png',
    'alpha': f'alpha_{SAMPLE_ID}.png',
    'depth': f'depth_{SAMPLE_ID}.exr',
    'normal': f'normal_{SAMPLE_ID}.exr',
}
for label, filename in modalities.items():
    path = SYNTHHUMAN_ROOT / filename
    start = time.perf_counter()
    try:
        exists = path.is_file()
        size = path.stat().st_size if exists else None
        elapsed = time.perf_counter() - start
        print(f'{label:>6}: exists={exists}, bytes={size}, metadata_seconds={elapsed:.3f}, path={path}')
    except OSError as exc:
        print(f'{label:>6}: OS ERROR after {time.perf_counter()-start:.3f}s: {type(exc).__name__}: {exc}')

In [ ]:
# Decode a single RGB, depth, and normal file to test actual reads, not just metadata.
from PIL import Image
import numpy as np

try:
    rgb_path = SYNTHHUMAN_ROOT / modalities['rgb']
    t0 = time.perf_counter()
    with Image.open(rgb_path) as image:
        rgb = np.asarray(image.convert('RGB'))
    print('RGB decoded:', rgb.shape, rgb.dtype, 'seconds=', round(time.perf_counter()-t0, 3))
except Exception as exc:
    print('RGB READ FAILED:', type(exc).__name__, str(exc))

# OpenEXR decoding is needed for depth and normals.
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
try:
    import cv2
    for label in ('depth', 'normal'):
        path = SYNTHHUMAN_ROOT / modalities[label]
        t0 = time.perf_counter()
        array = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if array is None:
            print(f'{label.upper()} DECODE FAILED: cv2.imread returned None ({time.perf_counter()-t0:.3f}s)')
        else:
            finite = np.isfinite(array)
            print(f'{label.upper()} decoded: shape={array.shape}, dtype={array.dtype}, finite_fraction={finite.mean():.4f}, seconds={time.perf_counter()-t0:.3f}')
except Exception as exc:
    print('OpenEXR test failed:', type(exc).__name__, str(exc))

In [ ]:
# Small sequential throughput sample: 8 known consecutive IDs, 3 modalities each.
# This is only a read test; it does not start training or scan the dataset.
import cv2
import numpy as np

NUM_SAMPLES = 8
FIRST_ID = int(SAMPLE_ID)
read_ok = 0
read_errors = []
t0 = time.perf_counter()
for sample_number in range(FIRST_ID, FIRST_ID + NUM_SAMPLES):
    sample_id = f'{sample_number:07d}'
    for filename in (f'rgb_{sample_id}.png', f'depth_{sample_id}.exr', f'normal_{sample_id}.exr'):
        try:
            path = SYNTHHUMAN_ROOT / filename
            if filename.endswith('.png'):
                with Image.open(path) as image:
                    _ = np.asarray(image.convert('RGB'))
            else:
                _ = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
                if _ is None:
                    raise ValueError('cv2.imread returned None')
            read_ok += 1
        except Exception as exc:
            read_errors.append((filename, type(exc).__name__, str(exc)))
            print('Read error:', read_errors[-1])
elapsed = time.perf_counter() - t0
print(f'Finished: {read_ok}/{NUM_SAMPLES*3} modality reads; elapsed={elapsed:.1f}s; reads_per_second={read_ok/max(elapsed,1e-9):.2f}')
print('Errors:', len(read_errors))

## Membaca hasil
- Jika cell 2 mendapat `OSError: [Errno 5]`, Drive FUSE masih gagal mengakses folder/sampel ini.
- Jika metadata berhasil tetapi decode RGB/EXR gagal, file atau dukungan OpenEXR perlu diperiksa.
- Jika seluruh read berhasil, itu membuktikan sampel kecil bisa diakses saja; belum membuktikan full training akan cukup cepat. Gunakan throughput cell terakhir sebagai dasar sebelum pilot kecil.
- Bagikan output cell 2–4 sebelum menjalankan training penuh.